# ⚡ ENTRENAMIENTO LLAMA-3.2-1B (ULTRA RÁPIDO)

---

## 📊 Especificaciones

- **Modelo:** Meta-Llama-3.2-1B-Instruct (1B parámetros)
- **Técnica:** QLoRA (4-bit)
- **GPU:** T4 (15GB VRAM) ✅
- **Tiempo:** 10-15 minutos ⚡⚡⚡
- **RAM:** ~4GB

---

## ✅ VENTAJAS

- ⚡ **MUY RÁPIDO** (10-15 min)
- ✅ **Poca RAM** (4GB)
- ✅ **No crashea** nunca
- ✅ **Calidad muy buena**
- ✅ **Llama 3.2** (2024)

---

In [ ]:
%%capture
!pip install -q --upgrade transformers peft accelerate bitsandbytes datasets sentencepiece einops
print("✅ Dependencias instaladas")

In [ ]:
import torch
print(f"✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No disponible'}")

In [ ]:
from google.colab import files
uploaded = files.upload()
print("✅ Dataset subido")

In [ ]:
from getpass import getpass
HF_TOKEN = getpass("Token HuggingFace: ")
print("✅ Token configurado")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/llama32_1b

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

# Cuantización
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# LoRA
lora_config = LoraConfig(
    r=8,  # Reducido para 1B
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Training
training_args = TrainingArguments(
    output_dir="./lora_model",
    num_train_epochs=10,
    per_device_train_batch_size=4,  # Batch más grande
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=True,
    logging_dir="./logs/llama32_1b",
    logging_steps=5,
    save_steps=100,
    warmup_steps=20,
    optim="paged_adamw_8bit",
    report_to="tensorboard",
    disable_tqdm=False,
    logging_first_step=True,
)

print("📚 Preparando dataset...")
with open('dataset_pedagogico.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

def format_instruction(example):
    text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Eres un asistente educativo experto. Responde SIEMPRE en español con tono pedagógico, motivador y amigable. Usa emojis apropiados.<|eot_id|><|start_header_id|>user<|end_header_id|>

{example['instruction']}

{example['input']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{example['output']}<|eot_id|>"""
    return {"text": text}

dataset = Dataset.from_list(data).map(format_instruction)
print(f"✅ {len(dataset)} ejemplos")

print("🤖 Cargando modelo...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    token=HF_TOKEN
)
print("✅ Modelo cargado")

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"📊 Entrenables: {trainable:,} ({100*trainable/total:.2f}%)")

print("📝 Tokenizando...")
def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

tokenized = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("✅ Dataset tokenizado")

print("\n🚀 INICIANDO ENTRENAMIENTO (10-15 min)\n")
trainer = Trainer(model=model, args=training_args, train_dataset=tokenized, data_collator=collator)
trainer.train()

print("\n✅ COMPLETADO")
print(f"📊 Loss final: {trainer.state.log_history[-1].get('loss', 'N/A')}")

model.save_pretrained("./lora_adapters")
tokenizer.save_pretrained("./lora_adapters")
print("💾 Adaptadores guardados")

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('lora_adapters_llama32_1b', 'zip', './lora_adapters')
files.download('lora_adapters_llama32_1b.zip')
print("✅ Descargado")
print("\n📋 Actualiza .env:")
print("HUGGINGFACE_MODEL=meta-llama/Llama-3.2-1B-Instruct")